# CNN Image Classification Use Case with CIFAR-10

## Overview
This notebook implements a **Convolutional Neural Network (CNN)** based image classification workflow using the **CIFAR-10** dataset. The notebook is documented in a clean, GitHub-friendly style so it can be used directly in a repository as a reproducible project artifact.

## Use Case
A compact CNN classifier like this can be used as the base for:
- **smart inventory monitoring**
- **basic object recognition in edge devices**
- **automated image tagging pipelines**
- **prototype visual quality inspection systems**

In this notebook, the model is trained to classify images into the following 10 categories:
`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`

## Workflow
1. Import dependencies
2. Load and inspect the dataset
3. Preprocess the images and labels
4. Build a LeNet-5 inspired CNN
5. Train the model
6. Evaluate performance
7. Visualize predictions
8. Translate the model into a practical use case

## Notes
- Dataset: **CIFAR-10**
- Framework: **TensorFlow / Keras**
- Model Type: **LeNet-5 inspired CNN**


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)

## 1. Load the Dataset

We use the CIFAR-10 dataset, which contains 60,000 color images of size **32 × 32** across 10 object classes. The dataset is split into training and testing sets.

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

## 2. Visual Inspection of Sample Images

Before training, it is good practice to visually inspect sample inputs. This helps confirm that the dataset is loaded correctly and gives intuition about the complexity of the classification problem.

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Preprocessing

The preprocessing steps are:
- normalize pixel values to the range **[0, 1]**
- convert class labels into **one-hot encoded vectors**

Since CIFAR-10 is already stored at **32 × 32**, resizing is not required for this model.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print("Processed x_train shape:", x_train.shape)
print("Processed x_test shape:", x_test.shape)
print("Encoded y_train shape:", y_train_cat.shape)
print("Encoded y_test shape:", y_test_cat.shape)

## 4. Model Architecture

This project uses a **LeNet-5 inspired CNN** adapted for RGB images. The architecture includes:
- convolution layers for feature extraction
- average pooling for spatial downsampling
- dense layers for classification
- dropout to reduce overfitting

This makes the model lightweight and suitable for learning-oriented or prototype use cases.

In [ ]:
lenet5 = models.Sequential(name="LeNet5_CIFAR10")

# Input layer
lenet5.add(layers.Input(shape=(32, 32, 3)))

# Feature extraction block 1
lenet5.add(layers.Conv2D(filters=6, kernel_size=(5, 5), activation="tanh", padding="same"))
lenet5.add(layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)))

# Feature extraction block 2
lenet5.add(layers.Conv2D(filters=16, kernel_size=(5, 5), activation="tanh"))
lenet5.add(layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)))

# Classification block
lenet5.add(layers.Flatten())
lenet5.add(layers.Dense(units=120, activation="tanh"))
lenet5.add(layers.Dropout(0.3))
lenet5.add(layers.Dense(units=84, activation="tanh"))
lenet5.add(layers.Dropout(0.2))
lenet5.add(layers.Dense(units=10, activation="softmax"))

lenet5.summary()

## 5. Compile the Model

We use:
- **Adam** optimizer for efficient training
- **categorical crossentropy** for multi-class classification
- **accuracy** as the primary performance metric

In [ ]:
lenet5.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## 6. Train the Model

The model is trained on the training split with a validation split to monitor generalization during training.

In [ ]:
history_lenet = lenet5.fit(
    x_train,
    y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 7. Evaluate the Model

We now measure final test performance on unseen data.

In [ ]:
test_loss, test_accuracy = lenet5.evaluate(x_test, y_test_cat, verbose=0)
print("LeNet-5 Test Loss     :", round(test_loss, 4))
print("LeNet-5 Test Accuracy :", round(test_accuracy, 4))

## 8. Training Curves

Loss and accuracy curves help determine whether the model is learning effectively and whether overfitting is emerging.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_lenet.history["loss"], label="Train Loss")
plt.plot(history_lenet.history["val_loss"], label="Validation Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_lenet.history["accuracy"], label="Train Accuracy")
plt.plot(history_lenet.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Classification Report and Confusion Matrix

A confusion matrix provides a detailed view of where the model performs well and where it confuses visually similar classes.

In [ ]:
y_pred_probs = lenet5.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Practical Use Case Demonstration

### Example Scenario: Smart Inventory Monitoring
A warehouse or retail system may receive a stream of product or transport images from cameras. A CNN classifier can act as the **first-stage vision model** to identify broad object categories before handing off the result to downstream systems.

### Why this matters
In a production pipeline, this model can support:
- automatic object tagging
- image routing by category
- alerting for unexpected object classes
- lightweight deployment for proof-of-concept computer vision systems

Below, we simulate inference by selecting sample test images and comparing the predicted label against the true label.

In [ ]:
sample_indices = np.random.choice(len(x_test), 10, replace=False)

plt.figure(figsize=(14, 6))
for i, idx in enumerate(sample_indices):
    image = x_test[idx]
    true_label = class_names[int(y_test[idx])]
    predicted_label = class_names[int(y_pred[idx])]

    plt.subplot(2, 5, i + 1)
    plt.imshow(image)
    plt.title(f"True: {true_label}
Pred: {predicted_label}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 11. Repository-Ready Summary

### Project Summary
This notebook demonstrates an end-to-end CNN image classification pipeline using CIFAR-10 and a LeNet-5 inspired architecture. The project covers data loading, preprocessing, model design, training, evaluation, and use-case interpretation.

### Key Takeaways
- CNNs can automatically learn spatial features from raw images.
- Even a relatively compact architecture can provide meaningful results on multi-class image data.
- The workflow is suitable as a baseline for more advanced computer vision projects.

### Suggested GitHub README Points
If you publish this project in a GitHub repository, the README can mention:
- project objective
- dataset details
- model architecture
- setup instructions
- training results
- real-world use cases
- future improvements such as batch normalization, data augmentation, and transfer learning
